# Notebook 03: Training Pipelines and Orchestration

## Why This Matters

Training a model locally in a notebook is not a production ML system. At companies like Spotify, Netflix, and Twitter, production models are retrained on schedules or triggered by data drift, with full lineage tracking, reproducibility guarantees, and automatic rollback. Tools like Kubeflow Pipelines, Metaflow, and Apache Airflow form the backbone of this infrastructure. Understanding DAG-based orchestration, how to handle retraining triggers, and the difference between scheduled and continuous training signals that you can build ML systems that stay accurate over time—not just systems that work once.

## 1. What Is a Training Pipeline?

A training pipeline is a directed acyclic graph (DAG) of steps that transforms raw data into a deployable model artifact. Each step has defined inputs, outputs, and failure modes. The pipeline is reproducible, auditable, and can be triggered manually or automatically.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import networkx as nx

fig, ax = plt.subplots(figsize=(16, 9))
ax.set_xlim(0, 16)
ax.set_ylim(0, 9)
ax.axis('off')
ax.set_title('Production ML Training Pipeline DAG', fontsize=14, fontweight='bold')

steps = [
    # (x, y, w, h, label, sublabel, color)
    (0.2, 7.2, 2.5, 1.2, 'Data Ingestion', 'Pull from DWH/S3', '#E74C3C'),
    (3.2, 7.2, 2.5, 1.2, 'Data Validation', 'Schema + stats checks', '#E67E22'),
    (6.2, 7.2, 2.5, 1.2, 'Feature Eng.', 'Transforms + joins', '#F39C12'),
    (9.2, 7.2, 2.5, 1.2, 'Train/Val Split', 'Time-based split', '#27AE60'),
    (12.2, 7.2, 2.5, 1.2, 'Baseline Check', 'Beat current prod', '#2980B9'),
    
    (0.2, 4.8, 2.5, 1.2, 'HP Tuning\n(optional)', 'Optuna / Ray Tune', '#8E44AD'),
    (3.2, 4.8, 2.5, 1.2, 'Model Training', 'Distributed if needed', '#6C3483'),
    (6.2, 4.8, 2.5, 1.2, 'Evaluation', 'Offline metrics suite', '#1ABC9C'),
    (9.2, 4.8, 2.5, 1.2, 'Bias/Fairness\nCheck', 'Slice analysis', '#16A085'),
    (12.2, 4.8, 2.5, 1.2, 'Model Registry\nPush', 'Artifact + metadata', '#2C3E50'),
    
    (0.2, 2.4, 2.5, 1.2, 'Shadow Deploy', 'No traffic impact', '#34495E'),
    (3.2, 2.4, 2.5, 1.2, 'A/B Test Launch', '5% traffic slice', '#2ECC71'),
    (6.2, 2.4, 2.5, 1.2, 'Online Eval', 'Monitor metrics', '#27AE60'),
    (9.2, 2.4, 2.5, 1.2, 'Promote to Prod', 'Gradual rollout', '#1A8754'),
    (12.2, 2.4, 2.5, 1.2, 'Monitor &\nAlert', 'Drift detection', '#E74C3C'),
]

for x, y, w, h, label, sub, color in steps:
    rect = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1",
                          facecolor=color, edgecolor='white', linewidth=1.8, alpha=0.88)
    ax.add_patch(rect)
    ax.text(x+w/2, y+h/2+0.15, label, ha='center', va='center',
            fontsize=8.5, fontweight='bold', color='white', multialignment='center')
    ax.text(x+w/2, y+h/2-0.3, sub, ha='center', va='center',
            fontsize=7.5, color='white', alpha=0.85)

# Row 1 arrows
for x in [2.7, 5.7, 8.7, 11.7]:
    ax.annotate('', xy=(x+0.5, 7.8), xytext=(x, 7.8),
                arrowprops=dict(arrowstyle='->', color='#aaa', lw=1.5))

# Row 2 arrows
for x in [2.7, 5.7, 8.7, 11.7]:
    ax.annotate('', xy=(x+0.5, 5.4), xytext=(x, 5.4),
                arrowprops=dict(arrowstyle='->', color='#aaa', lw=1.5))

# Row 3 arrows
for x in [2.7, 5.7, 8.7, 11.7]:
    ax.annotate('', xy=(x+0.5, 3.0), xytext=(x, 3.0),
                arrowprops=dict(arrowstyle='->', color='#aaa', lw=1.5))

# Row 1 -> Row 2 (end to start)
ax.annotate('', xy=(1.5, 6.0), xytext=(13.45, 7.2),
            arrowprops=dict(arrowstyle='->', color='#666', lw=1.5,
                            connectionstyle='arc3,rad=0.0'))

# Row 2 -> Row 3 (end to start)
ax.annotate('', xy=(1.5, 3.6), xytext=(13.45, 4.8),
            arrowprops=dict(arrowstyle='->', color='#666', lw=1.5))

# Row labels
for y, label in [(7.8, 'Data Phase'), (5.4, 'Training Phase'), (3.0, 'Deployment Phase')]:
    ax.text(15.8, y, label, ha='right', va='center', fontsize=9, color='#888', rotation=270)

plt.tight_layout()
plt.savefig('training_pipeline_dag.png', dpi=120, bbox_inches='tight')
plt.show()

## 2. Orchestration Frameworks Comparison

Choosing an orchestration tool is a key design decision. The choice affects developer experience, scalability, and operational complexity. Airflow (workflow scheduling), Kubeflow (Kubernetes-native ML), and Metaflow (data scientist-friendly) each occupy different niches.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

tools = ['Apache\nAirflow', 'Kubeflow\nPipelines', 'Metaflow\n(Netflix)', 'Prefect', 'ZenML', 'Vertex AI\nPipelines']

# Score 1-5 across dimensions
dimensions = [
    'DS Friendliness',
    'Scale (K8s)',
    'ML-Specific\nFeatures',
    'Ecosystem\nMaturity',
    'Cloud-Native\nIntegration',
    'Open Source\nFlexibility',
]

scores = {
    'Apache\nAirflow':       [2, 3, 2, 5, 4, 5],
    'Kubeflow\nPipelines':   [2, 5, 5, 4, 4, 4],
    'Metaflow\n(Netflix)':   [5, 4, 4, 3, 3, 5],
    'Prefect':               [4, 3, 3, 3, 4, 5],
    'ZenML':                 [4, 4, 4, 2, 4, 5],
    'Vertex AI\nPipelines':  [3, 5, 5, 3, 5, 2],
}

colors = ['#E74C3C', '#3498DB', '#E67E22', '#9B59B6', '#27AE60', '#F39C12']

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Radar chart
ax = axes[0]
n = len(dimensions)
angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
angles += angles[:1]

ax_polar = plt.subplot(121, polar=True)
for (tool, s), color in zip(scores.items(), colors):
    values = s + s[:1]
    ax_polar.plot(angles, values, 'o-', linewidth=1.5, color=color, alpha=0.8, label=tool.replace('\n', ' '))
    ax_polar.fill(angles, values, alpha=0.08, color=color)

ax_polar.set_xticks(angles[:-1])
ax_polar.set_xticklabels(dimensions, fontsize=8)
ax_polar.set_ylim(0, 5)
ax_polar.set_title('Orchestration Tool Comparison', fontsize=11, fontweight='bold', pad=20)
ax_polar.legend(loc='upper right', bbox_to_anchor=(1.4, 1.1), fontsize=8)
ax_polar.set_yticks([1, 2, 3, 4, 5])
ax_polar.set_yticklabels(['1', '2', '3', '4', '5'], fontsize=7)
ax_polar.grid(True, alpha=0.3)

# Heatmap
ax2 = axes[1]
data = np.array([scores[t] for t in tools])
im = ax2.imshow(data, cmap='RdYlGn', vmin=1, vmax=5, aspect='auto')
ax2.set_xticks(range(len(dimensions)))
ax2.set_xticklabels([d.replace('\n', ' ') for d in dimensions], rotation=30, ha='right', fontsize=8)
ax2.set_yticks(range(len(tools)))
ax2.set_yticklabels([t.replace('\n', ' ') for t in tools], fontsize=9)
ax2.set_title('Score Matrix (1=low, 5=high)', fontsize=11, fontweight='bold')

for i in range(len(tools)):
    for j in range(len(dimensions)):
        ax2.text(j, i, str(data[i, j]), ha='center', va='center',
                 fontsize=10, fontweight='bold',
                 color='white' if data[i, j] < 3 else '#333')

plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig('orchestration_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

print("Recommendation guide:")
recs = [
    ("Airflow", "Data engineering heavy team, existing Airflow infra, general workflows"),
    ("Kubeflow", "Kubernetes-first org, need GPU scheduling, enterprise ML platform"),
    ("Metaflow", "Data science team, need versioning + local dev, AWS/GCP shop"),
    ("Vertex AI Pipelines", "GCP shop, serverless execution, managed infra preferred"),
]
for tool, when in recs:
    print(f"  {tool}: {when}")

## 3. Retraining Strategies

When to retrain is as important as how to retrain. Unnecessary retraining wastes compute; infrequent retraining causes model decay. The right strategy depends on how quickly your data distribution changes.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

np.random.seed(42)
days = np.arange(0, 90)

# Simulate model performance decay over time
def model_perf(deploy_day, days_arr, decay_rate=0.008, noise=0.01):
    """Model performance decays from deployment day until retrained."""
    elapsed = days_arr - deploy_day
    elapsed = np.clip(elapsed, 0, None)
    perf = 0.85 * np.exp(-decay_rate * elapsed) + np.random.normal(0, noise, len(days_arr))
    perf = np.where(days_arr < deploy_day, np.nan, perf)
    return perf

fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=True)
strategies = [
    ("Scheduled (Weekly)", [0, 7, 14, 21, 28, 35, 42, 49, 56, 63, 70, 77, 84]),
    ("Scheduled (Monthly)", [0, 30, 60]),
    ("Trigger-Based", [0, 15, 28, 60, 75]),  # triggered by drift
]

for ax, (strategy, retrain_days) in zip(axes, strategies):
    # Plot each model segment
    overall_perf = np.full(len(days), np.nan)
    
    for i, deploy_day in enumerate(retrain_days):
        end_day = retrain_days[i+1] if i+1 < len(retrain_days) else 90
        segment_days = np.arange(deploy_day, end_day)
        if len(segment_days) == 0:
            continue
        perf = 0.85 * np.exp(-0.008 * (segment_days - deploy_day)) + np.random.normal(0, 0.005, len(segment_days))
        color = plt.cm.Blues(0.5 + i * 0.1)
        ax.plot(segment_days, perf, color=color, linewidth=2, alpha=0.9)
        overall_perf[deploy_day:end_day] = perf
        ax.axvline(x=deploy_day, color='red', linestyle='--', alpha=0.7, linewidth=1.2)
        ax.text(deploy_day + 0.5, 0.84, f'retrain', fontsize=7, color='red', rotation=90, va='top')

    # Threshold line
    ax.axhline(y=0.75, color='orange', linestyle=':', linewidth=1.5, label='Alert threshold')
    ax.axhline(y=0.70, color='red', linestyle=':', linewidth=1.5, label='Hard floor')

    # Avg perf
    valid = overall_perf[~np.isnan(overall_perf)]
    avg = np.nanmean(valid) if len(valid) > 0 else 0
    retrains = len(retrain_days)

    ax.set_title(f'{strategy}\n(avg perf: {avg:.3f}, retrains: {retrains})',
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('Day', fontsize=10)
    ax.set_ylabel('Model Performance (AUC)', fontsize=10)
    ax.set_ylim(0.65, 0.92)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, loc='lower left')

plt.suptitle('Retraining Strategies: Performance vs Cost', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('retraining_strategies.png', dpi=120, bbox_inches='tight')
plt.show()

print("Retraining Strategy Comparison:")
print(f"  Weekly:  High compute cost, stable performance, predictable")
print(f"  Monthly: Low cost, performance degrades between retrains")
print(f"  Trigger: Best efficiency, requires drift detection infrastructure")

## 4. Pipeline Components: Data Validation

Data validation is the most commonly skipped step in amateur pipelines and the most critical in production. Silent data quality issues—missing columns, distribution shifts, schema changes—cause model degradation that looks identical to model decay. TensorFlow Data Validation (TFDV) and Great Expectations are the leading tools.

In [ ]:
import numpy as np
import pandas as pd
from typing import Dict, Any

class DataValidationStep:
    """
    Lightweight data validation step for a training pipeline.
    Production equivalents: TensorFlow Data Validation, Great Expectations, Deequ.
    """
    
    def __init__(self, schema: Dict[str, Any], reference_stats: Dict[str, Any] = None):
        self.schema = schema
        self.reference_stats = reference_stats or {}
        self.anomalies = []
    
    def validate(self, df: pd.DataFrame) -> bool:
        """Run all validations. Returns True if pipeline should proceed."""
        self.anomalies = []
        
        self._check_schema(df)
        self._check_missing_rate(df)
        self._check_distribution_drift(df)
        self._check_label_balance(df)
        self._check_min_rows(df)
        
        return len([a for a in self.anomalies if a['severity'] == 'ERROR']) == 0
    
    def _check_schema(self, df):
        required_cols = self.schema.get('required_columns', [])
        for col in required_cols:
            if col not in df.columns:
                self.anomalies.append({
                    'check': 'schema', 'severity': 'ERROR',
                    'message': f'Required column missing: {col}'
                })
    
    def _check_missing_rate(self, df):
        max_missing = self.schema.get('max_missing_fraction', 0.1)
        for col in df.columns:
            missing_rate = df[col].isna().mean()
            if missing_rate > max_missing:
                severity = 'ERROR' if missing_rate > 0.5 else 'WARNING'
                self.anomalies.append({
                    'check': 'missing_rate', 'severity': severity,
                    'message': f'{col}: {missing_rate:.1%} missing (threshold: {max_missing:.0%})'
                })
    
    def _check_distribution_drift(self, df):
        for col, ref in self.reference_stats.items():
            if col not in df.columns:
                continue
            current_mean = df[col].mean()
            ref_mean = ref['mean']
            ref_std = ref['std']
            z_score = abs(current_mean - ref_mean) / (ref_std + 1e-8)
            if z_score > 5:
                self.anomalies.append({
                    'check': 'distribution_drift', 'severity': 'ERROR',
                    'message': f'{col}: mean drifted by {z_score:.1f} std devs (current={current_mean:.3f}, ref={ref_mean:.3f})'
                })
            elif z_score > 3:
                self.anomalies.append({
                    'check': 'distribution_drift', 'severity': 'WARNING',
                    'message': f'{col}: possible drift, z={z_score:.1f}'
                })
    
    def _check_label_balance(self, df):
        label_col = self.schema.get('label_column')
        if label_col and label_col in df.columns:
            pos_rate = df[label_col].mean()
            if pos_rate < 0.001 or pos_rate > 0.999:
                self.anomalies.append({
                    'check': 'label_balance', 'severity': 'ERROR',
                    'message': f'Extreme label imbalance: {pos_rate:.4%} positive rate'
                })
    
    def _check_min_rows(self, df):
        min_rows = self.schema.get('min_rows', 1000)
        if len(df) < min_rows:
            self.anomalies.append({
                'check': 'row_count', 'severity': 'ERROR',
                'message': f'Only {len(df)} rows (minimum: {min_rows})'
            })
    
    def report(self):
        if not self.anomalies:
            print("  ✓ All validation checks passed")
            return
        errors = [a for a in self.anomalies if a['severity'] == 'ERROR']
        warnings = [a for a in self.anomalies if a['severity'] == 'WARNING']
        print(f"  {len(errors)} ERRORs, {len(warnings)} WARNINGs:")
        for a in self.anomalies:
            icon = '✗' if a['severity'] == 'ERROR' else '⚠'
            print(f"    {icon} [{a['check']}] {a['message']}")


# Demonstrate validation on clean vs corrupt data
schema = {
    'required_columns': ['user_id', 'item_id', 'feature_a', 'feature_b', 'clicked'],
    'max_missing_fraction': 0.1,
    'label_column': 'clicked',
    'min_rows': 100,
}
ref_stats = {
    'feature_a': {'mean': 0.5, 'std': 0.1},
    'feature_b': {'mean': 10.0, 'std': 2.0},
}

np.random.seed(42)
n = 5000

# Clean data
clean_df = pd.DataFrame({
    'user_id': range(n), 'item_id': range(n),
    'feature_a': np.random.normal(0.5, 0.1, n),
    'feature_b': np.random.normal(10.0, 2.0, n),
    'clicked': np.random.binomial(1, 0.05, n),
})

# Corrupt data: missing column, high missing rate, distribution drift
corrupt_df = clean_df.copy()
corrupt_df = corrupt_df.drop(columns=['item_id'])  # missing column
corrupt_df.loc[:500, 'feature_a'] = np.nan  # 10% missing
corrupt_df['feature_b'] = np.random.normal(25.0, 2.0, n)  # distribution drift!

validator = DataValidationStep(schema, ref_stats)

print("=" * 55)
print("Validating CLEAN training data:")
print("=" * 55)
clean_ok = validator.validate(clean_df)
validator.report()
print(f"  → Pipeline proceed: {clean_ok}")

print("\n" + "=" * 55)
print("Validating CORRUPT training data:")
print("=" * 55)
corrupt_ok = validator.validate(corrupt_df)
validator.report()
print(f"  → Pipeline proceed: {corrupt_ok}")

## 5. Experiment Tracking and Reproducibility

Without experiment tracking, it's impossible to reproduce results, understand which changes improved performance, or audit model decisions. MLflow and Weights & Biases are the dominant tools. The key metadata to track: data version, code commit, hyperparameters, environment, and all metrics.

In [ ]:
import json
import hashlib
from datetime import datetime
import pandas as pd

class ExperimentTracker:
    """
    Lightweight experiment tracker.
    Production: MLflow, Weights & Biases, Neptune, Comet.
    """
    
    def __init__(self):
        self.runs = []
        self._current_run = None
    
    def start_run(self, experiment_name: str, tags: dict = None):
        run_id = hashlib.md5(f"{experiment_name}{datetime.now()}".encode()).hexdigest()[:8]
        self._current_run = {
            'run_id': run_id,
            'experiment': experiment_name,
            'start_time': datetime.now().isoformat()[:19],
            'tags': tags or {},
            'params': {},
            'metrics': {},
            'artifacts': [],
            'status': 'RUNNING',
        }
        return run_id
    
    def log_param(self, key: str, value):
        self._current_run['params'][key] = value
    
    def log_params(self, params: dict):
        self._current_run['params'].update(params)
    
    def log_metric(self, key: str, value: float):
        self._current_run['metrics'][key] = round(value, 6)
    
    def log_metrics(self, metrics: dict):
        for k, v in metrics.items():
            self.log_metric(k, v)
    
    def log_artifact(self, path: str):
        self._current_run['artifacts'].append(path)
    
    def end_run(self, status='FINISHED'):
        self._current_run['status'] = status
        self._current_run['end_time'] = datetime.now().isoformat()[:19]
        self.runs.append(self._current_run)
        return self._current_run
    
    def compare_runs(self) -> pd.DataFrame:
        rows = []
        for run in self.runs:
            row = {'run_id': run['run_id'], 'experiment': run['experiment'], 'status': run['status']}
            row.update({f'param_{k}': v for k, v in run['params'].items()})
            row.update({f'metric_{k}': v for k, v in run['metrics'].items()})
            rows.append(row)
        return pd.DataFrame(rows)


tracker = ExperimentTracker()

# Simulate 4 experiment runs
experiments = [
    {'name': 'ranking_model_v1', 'params': {'lr': 0.001, 'epochs': 5, 'hidden': [128, 64], 'dropout': 0.3},
     'metrics': {'auc': 0.782, 'ndcg_10': 0.421, 'train_loss': 0.312, 'val_loss': 0.334}},
    {'name': 'ranking_model_v2', 'params': {'lr': 0.0005, 'epochs': 10, 'hidden': [256, 128, 64], 'dropout': 0.2},
     'metrics': {'auc': 0.798, 'ndcg_10': 0.437, 'train_loss': 0.298, 'val_loss': 0.318}},
    {'name': 'ranking_model_v3', 'params': {'lr': 0.001, 'epochs': 10, 'hidden': [256, 128, 64], 'dropout': 0.4},
     'metrics': {'auc': 0.801, 'ndcg_10': 0.445, 'train_loss': 0.305, 'val_loss': 0.315}},
    {'name': 'ranking_model_v3_retrain', 'params': {'lr': 0.001, 'epochs': 10, 'hidden': [256, 128, 64], 'dropout': 0.4},
     'metrics': {'auc': 0.803, 'ndcg_10': 0.448, 'train_loss': 0.301, 'val_loss': 0.312}},
]

for exp in experiments:
    tracker.start_run(exp['name'], tags={'team': 'ranking', 'data_version': 'v2024-01'})
    tracker.log_params(exp['params'])
    tracker.log_metrics(exp['metrics'])
    tracker.log_artifact(f"models/{exp['name']}.pkl")
    tracker.end_run()

comparison = tracker.compare_runs()
print("Experiment Comparison:")
print("=" * 80)
key_cols = ['run_id', 'experiment', 'metric_auc', 'metric_ndcg_10', 'metric_val_loss', 'param_lr', 'param_dropout']
print(comparison[key_cols].to_string(index=False))

best_run = comparison.loc[comparison['metric_auc'].idxmax()]
print(f"\nBest run: {best_run['experiment']} (AUC={best_run['metric_auc']:.3f})")

# Key metadata for reproducibility
print("\nReproducibility checklist for production:")
checklist = [
    "✓ Data snapshot hash / version tag logged",
    "✓ Code git commit SHA logged",
    "✓ All hyperparameters logged (not just the good ones)",
    "✓ Python + library versions (requirements.txt or conda env)",
    "✓ Random seed logged",
    "✓ Hardware spec logged (GPU type, memory)",
    "✓ Training time and cost logged",
]
for item in checklist:
    print(f"  {item}")

## 6. Continuous Training vs. Periodic Training

Continuous training keeps models fresh by incorporating new data incrementally. It reduces the training data batch size but requires careful handling of catastrophic forgetting and concept drift detection.

In [ ]:
comparison_table = {
    'Strategy': ['Periodic (Weekly)', 'Periodic (Daily)', 'Triggered (Drift)', 'Continuous (Online)', 'Warm-start (Incremental)'],
    'Training Data': ['Full history', 'Full history', 'Full history', 'Mini-batches', 'Recent window'],
    'Freshness': ['Days stale', 'Hours stale', 'Triggered by drift', 'Minutes stale', 'Hours stale'],
    'Compute Cost': ['Medium', 'High', 'Medium (variable)', 'Low per run, high total', 'Low-Medium'],
    'Complexity': ['Low', 'Low', 'Medium (drift detector)', 'High (stability)', 'Medium'],
    'Catastrophic\nForgetting Risk': ['None', 'None', 'None', 'High', 'Low (replay buffer)'],
    'Best For': [
        'Stable data distributions',
        'Moderate drift, affordable compute',
        'Unknown drift schedule',
        'High-velocity streams (ads, fraud)',
        'Gradual drift, cost-sensitive',
    ]
}

df = pd.DataFrame(comparison_table)
print(df.to_string(index=False))

print("\n" + "="*60)
print("Production examples:")
print("="*60)
examples = [
    ("Google Ads CTR model", "Continuous training on click streams — very high velocity data"),
    ("Spotify Discover Weekly", "Weekly batch retrain — aligns with the product's weekly cadence"),
    ("Fraud detection (Stripe)", "Triggered retraining + rule updates when new fraud patterns detected"),
    ("Netflix recommendations", "Daily retrain on new watch history + content catalog updates"),
    ("Uber surge pricing ML", "Continuous — must reflect real-time supply/demand"),
]
for company, strategy in examples:
    print(f"  {company}: {strategy}")

## Summary

| Concept | Key Point | Interview Signal |
|---------|-----------|------------------|
| DAG-based pipelines | Training is a graph of steps, not a script | Can enumerate the steps and their dependencies |
| Data validation | Check schema, missing rates, distribution drift *before* training | Knows what breaks silently without this |
| Orchestration tools | Airflow (general), Kubeflow (K8s), Metaflow (DS-friendly) | Can justify tool choice for the given context |
| Retraining triggers | Scheduled, drift-triggered, or continuous | Knows tradeoffs in freshness vs cost |
| Experiment tracking | Params + metrics + data version + code SHA | Can explain what's needed for reproducibility |
| Continuous training | Online learning for high-velocity data; risk of catastrophic forgetting | Knows failure modes, not just benefits |
| Training-serving skew | Pipeline must match serving code exactly | Often the root cause of production ML failures |